In [24]:
# 三种meta-feature方法：TSFEL，TSFused，Foundation models
import tsfel
import numpy as np
import pandas as pd
import warnings
import tsfel
import os
from tqdm import tqdm
from pathlib import Path
import sklearn as sk
from scipy.stats import skew, kurtosis, entropy
from scipy.signal import periodogram
from statsmodels.tsa.stattools import acf, adfuller
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.ar_model import AutoReg
from tabpfn_extensions import TabPFNClassifier
from tabpfn_extensions.embedding import TabPFNEmbedding
print("TabPFN Extensions imported successfully.")
import warnings
warnings.filterwarnings("ignore")

TabPFN Extensions imported successfully.


In [25]:
# 读取数据
root_path = "/data/nishome/user1/chaochuan/TSGym_benchmark/dataset"
dataset_dir = [x for x in os.listdir(root_path) if 'ETT' not in x and 'plots_multivariate' not in x]
root_dir = Path(root_path)
file_paths = [str(p) for p in root_dir.rglob('*') if p.is_file() and str(p).endswith('.csv') and 'plots' not in str(p) and 'm4' not in str(p) and '00' not in str(p)]

In [26]:
file_paths

['/data/nishome/user1/chaochuan/TSGym_benchmark/dataset/czelan/czelan.csv',
 '/data/nishome/user1/chaochuan/TSGym_benchmark/dataset/wind/wind.csv',
 '/data/nishome/user1/chaochuan/TSGym_benchmark/dataset/exchange_rate/exchange_rate.csv',
 '/data/nishome/user1/chaochuan/TSGym_benchmark/dataset/metr-la/metr-la.csv',
 '/data/nishome/user1/chaochuan/TSGym_benchmark/dataset/pems08/pems08.csv',
 '/data/nishome/user1/chaochuan/TSGym_benchmark/dataset/fred-md/fred-md.csv',
 '/data/nishome/user1/chaochuan/TSGym_benchmark/dataset/aqwan/aqwan.csv',
 '/data/nishome/user1/chaochuan/TSGym_benchmark/dataset/finance_nasdaq/finance_nasdaq.csv',
 '/data/nishome/user1/chaochuan/TSGym_benchmark/dataset/nyse/nyse.csv',
 '/data/nishome/user1/chaochuan/TSGym_benchmark/dataset/nasdaq/nasdaq.csv',
 '/data/nishome/user1/chaochuan/TSGym_benchmark/dataset/ETT-small/ETTm1.csv',
 '/data/nishome/user1/chaochuan/TSGym_benchmark/dataset/ETT-small/ETTh1.csv',
 '/data/nishome/user1/chaochuan/TSGym_benchmark/dataset/ETT-

In [27]:
def read_data(file_path):
    """
    read data from file_path, return training set
    """
    # check
    if not os.path.isfile(file_path):
        assert FileNotFoundError
    data = pd.read_csv(file_path, header=0)
    data = data.dropna(axis=1, how='all')  # in case there is a column with all nans
    train_ratio=0.6 if 'ETT' in file_path else 0.7 # 0.6 for ETT and PEMS
    train_length = int(data.shape[0]*train_ratio)
    data = data[train_length-int(data.shape[0]*train_ratio):train_length]
    data = data.drop(columns=['date','Date','timestamp'], errors='ignore')
    return data

In [28]:
# data_list = [read_data(_) for _ in file_paths]

In [32]:
# 基于TSFEL提取meta-features
from sklearn.random_projection import GaussianRandomProjection

def get_meta_feature_tsfel(file_path, target_dim=2000):
    data = read_data(file_path)
    data = data.values.astype(np.float32) # tsfel要求输入为numpy array
    
    # get meat feature
    # Extracts the temporal, statistical and spectral feature sets.
    # Returns a DataFrame with the features.
    cfg = tsfel.get_features_by_domain() 
    
    meta_feature_list = []
    # Iterate over each variable (column) in the time series
    for i in range(data.shape[1]):
        # Extract features for the single time series
        # Returns shape (1, n_features)
        X = tsfel.time_series_features_extractor(cfg, data[:,i], fs=100, verbose=0).values
        meta_feature_list.append(X)
    
    # Concatenate to shape (n_variables, n_features)
    meta_feature = np.concatenate(meta_feature_list, axis=0)
    
    # Aggregate over variables using 9 statistics to capture the distribution of features across the variables
    # This results in shape (9, n_features)
    mean = np.mean(meta_feature, axis=0)
    std = np.std(meta_feature, axis=0)
    min_val = np.min(meta_feature, axis=0)
    q25 = np.percentile(meta_feature, 25, axis=0)
    median = np.median(meta_feature, axis=0)
    q75 = np.percentile(meta_feature, 75, axis=0)
    max_val = np.max(meta_feature, axis=0)
    range_val = max_val - min_val
    iqr = q75 - q25
    
    combined_features = np.stack([mean, std, min_val, q25, median, q75, max_val, range_val, iqr])
    # combined_features shape is approx (9, 156) ~ 1404 dimensions flattened
    
    # Flatten high-dimensional feature matrix
    features_flat = combined_features.flatten().reshape(1, -1)
    
    # Reduce dimensionality to target_dim (200) using Gaussian Random Projection
    # A fixed random_state ensures the projection matrix is the same for every dataset,
    # satisfying the requirement of consistency without using other datasets' data.
    if features_flat.shape[1] > target_dim:
        transformer = GaussianRandomProjection(n_components=target_dim, random_state=42)
        features_reduced = transformer.fit_transform(features_flat)
        return features_reduced.flatten()
    else:
        return features_flat.flatten()


In [7]:
# features = get_meata_feature_tsfel(file_paths[0],200)

In [30]:
# 基于TSFused提取meta-features
def get_meta_feature_tsfused(file_path):
    """
    Extracts meta-features from a given time series data.

    Parameters:
    - data: np.ndarray, shape (n_samples, n_features), time series data

    Returns:
    - features: dict, contains the extracted meta-features
    """
    data = read_data(file_path)
    data = data.values.astype(np.float32) # tsfel要求输入为numpy array
    features = {}

    # basic statistics
    features["mean"] = np.mean(data, axis=0).mean()
    features["std"] = np.std(data, axis=0).mean()
    features["min"] = np.min(data, axis=0).mean()
    features["max"] = np.max(data, axis=0).mean()
    features["skewness"] = np.nanmean(skew(data, axis=0))
    features["kurtosis"] = np.nanmean(kurtosis(data, axis=0))

    # time series decomposition
    acfs = [acf(data[:, i], nlags=10, fft=True) for i in range(data.shape[1])]
    features["autocorrelation_mean"] = np.nanmean(
        [acf_val[1] for acf_val in acfs]
    )  # first lag
    adf_results = [adfuller(data[:, i]) for i in range(data.shape[1])]
    features["stationarity"] = np.mean([result[1] < 0.05 for result in adf_results])

    # rate_of_change = np.diff(data, axis=0) / data[:-1]
    # Deal with 0 division
    safe_data = np.where(data[:-1] == 0, np.nan, data[:-1])
    rate_of_change = np.diff(data, axis=0) / safe_data
    features["rate_of_change_mean"] = np.nanmean(rate_of_change)
    features["rate_of_change_std"] = np.nanstd(rate_of_change)

    # Landmarker features
    autoreg_coefs, residual_stds = [], []
    for i in range(data.shape[1]):
        model = AutoReg(data[:, i], lags=1).fit()
        autoreg_coefs.append(model.params[1])
        residual_stds.append(np.std(model.resid))
    features["autoreg_coef_mean"] = np.mean(autoreg_coefs)
    features["residual_std_mean"] = np.mean(residual_stds)

    # frequency domain features
    freq_means, freq_peaks, spectral_entropies = [], [], []
    spectral_variations, spectral_skewnesses, spectral_kurtoses = [], [], []

    for i in range(data.shape[1]):
        freqs, psd = periodogram(data[:, i])
        freq_means.append(np.mean(psd))
        freq_peaks.append(freqs[np.argmax(psd)])
        spectral_entropies.append(entropy(psd))
        if i > 0:
            prev_psd = periodogram(data[:, i - 1])[1]
            spectral_variations.append(np.sqrt(np.sum((psd - prev_psd) ** 2)))
        else:
            spectral_variations.append(0)  # 第一个变量无法计算变化
        spectral_skewnesses.append(skew(psd))
        spectral_kurtoses.append(kurtosis(psd))

    features["frequency_mean"] = np.mean(freq_means)
    features["frequency_peak"] = np.mean(freq_peaks)
    features["spectral_entropy"] = np.nanmean(spectral_entropies)
    features["spectral_variation"] = np.nanmean(spectral_variations)
    features["spectral_skewness"] = np.nanmean(spectral_skewnesses)
    features["spectral_kurtosis"] = np.nanmean(spectral_kurtoses)

    cov_matrix = np.cov(data, rowvar=False)
    features["covariance_mean"] = np.mean(cov_matrix)
    features["covariance_max"] = np.max(cov_matrix)
    features["covariance_min"] = np.min(cov_matrix)
    features["covariance_std"] = np.std(cov_matrix)
    # dict to numpy array
    features = np.array(list(features.values()))
    return features

In [9]:
# 利用TabPFN提取序列Embedding

In [31]:
# 构造自监督任务 (Self-Supervised Task) 用于 TabPFN
# 由于 TabPFN 是分类模型，我们可以通过"预测下一个时间步的值（离散化后）"来构造标签
def prepare_tabpfn_data(file_path, window_len=50, n_samples=None, n_classes=10):
    """
    构造 (N, T) 的样本和 (N,) 的分类标签
    """
    df = read_data(file_path) # 使用前面定义的 read_data
    data = df.values
    if n_samples is None:
        n_samples = max(2000, data.shape[0] // 2)
    
    # 简单标准化，防止某些序列数值过大
    data = (data - np.nanmean(data, axis=0)) / (np.nanstd(data, axis=0) + 1e-8)
    
    X_list = []
    y_list = []
    
    n_timesteps, n_features = data.shape
    
    # 随机采样窗口
    # 如果数据量不够，就遍历所有
    possible_starts = n_timesteps - window_len - 1
    if possible_starts <= 0:
        return np.zeros((0, window_len)), np.zeros((0,))

    for _ in range(n_samples):
        # 随机选一个变量
        feat_idx = np.random.randint(0, n_features)
        # 随机选一个起始点
        start_idx = np.random.randint(0, possible_starts)
        
        window = data[start_idx : start_idx + window_len, feat_idx]
        target = data[start_idx + window_len, feat_idx]
        
        X_list.append(window)
        y_list.append(target)
        
    X = np.stack(X_list)
    y_continuous = np.array(y_list)
    
    # 将连续目标离散化为类别 (Binning)
    # 使用分位数分桶，保证类别平衡
    try:
        y = pd.qcut(y_continuous, q=n_classes, labels=False, duplicates='drop')
    except ValueError:
        # 如果数据过于集中导致分位数重复，使用等宽分桶
        y = pd.cut(y_continuous, bins=n_classes, labels=False)
    
    # 处理可能的 NaN (pd.cut 可能会产生 NaN)
    y = np.nan_to_num(y, nan=0).astype(int)
        
    return X, y

def get_tabpfn_embedding(file_path, window_len=50, n_samples=None, n_classes=10):
    """
    使用 TabPFN 提取时间序列的 Embedding
    """
    X, y = prepare_tabpfn_data(file_path, window_len, n_samples, n_classes)
    if X.shape[0] == 0:
        # 如果没有样本，返回全零向量
        return np.zeros((128,))
    
    # 初始化 TabPFN Embedding 模型
    model_path = "/data/nishome/user1/xwyl/llm/tabfpn-v2/tabpfn-v2.5-classifier-v2.5_default.ckpt"
    classifier = TabPFNClassifier(device="cuda", n_estimators=4, model_path=model_path)
    embedding_extractor = TabPFNEmbedding(tabpfn_clf=classifier, n_fold=5)

    
    # 获取 Embedding
    train_embeddings_full = embedding_extractor.get_embeddings(X, y, X, data_source="train")

    X_train_emb = train_embeddings_full.mean(axis=0)
    
    # 对所有样本的 Embedding 求均值，得到固定长度的特征向量
    
    dataset_meta_feature = X_train_emb.mean(axis=0)
    
    return dataset_meta_feature

In [1]:
def get_meta_faetures(meta_feature_type='tsfel',save_path=None):
    if meta_feature_type == 'tsfel':
        meta_features = get_meta_feature_tsfel(file_path)
        save_path = save_path if save_path is not None else "../meta/meta_features/meta_feature_dict_tsfel.npz"
    elif meta_feature_type == 'tsfel_gaussianRandomProjection':
        meta_features = get_meta_feature_tsfel(file_path, target_dim=256)
        save_path = save_path if save_path is not None else "../meta/meta_features/meta_feature_dict_tsfelGRP.npz"
    elif meta_feature_type == 'tsfused':
        meta_features = get_meta_feature_tsfused(file_path)
        save_path = save_path if save_path is not None else "../meta/meta_features/meta_feature_dict_tsfused.npz"
    elif meta_feature_type == 'tabpfn':
        meta_features = get_tabpfn_embedding(file_path)
        save_path = save_path if save_path is not None else "../meta/meta_features/meta_feature_dict_tabpfn.npz"
    else:
        raise ValueError("Unknown meta_feature_type: {}".format(meta_feature_type))
    np.savez(save_path, **meta_features)
    return meta_features

In [ ]:
dict_meta_features_tsfel = {_.split('/')[-1].split('.')[0]: get_meta_feature_tsfel(_) for _ in tqdm(file_paths)}

100%|██████████| 23/23 [09:50<00:00, 25.68s/it]


In [21]:
dict_meta_features_tsfel.keys()

dict_keys(['czelan', 'wind', 'exchange_rate', 'metr-la', 'pems08', 'fred-md', 'aqwan', 'finance_nasdaq', 'nyse', 'nasdaq', 'nn5', 'covid-19', 'weather', 'us_births_dataset_1', 'traffic', 'pems04', 'solar', 'illness', 'zafnoo', 'pems-bay', 'finance_nyse', 'aqshunyi', 'electricity'])

In [ ]:
dict_meta_features_tsfel_gaussianRandomProjection = {_.split('/')[-2]: get_meta_feature_tsfel(_, target_dim=256) for _ in tqdm(file_paths)}

In [33]:
dict_meta_features_tsfused = {_.split('/')[-2]: get_meta_feature_tsfused(_) for _ in tqdm(file_paths)}

 11%|█         | 3/27 [00:21<02:36,  6.53s/it]

KeyboardInterrupt: 

In [ ]:
dict_meta_features_tabpfn = {_.split('/')[-2]: get_tabpfn_embedding(_) for _ in tqdm(file_paths)}